In [1]:
from pathlib import Path
import pandas as pd

# Define dataset path
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

MEDIA_PATH = DATA_DIR / "media.json"

if not MEDIA_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {MEDIA_PATH}"
    )

# Load original dataset
df_media_raw = pd.read_json(MEDIA_PATH)

# Create working copy
df_media_clean = df_media_raw.copy()

print("Original dataset shape:", df_media_raw.shape)

print("\nOriginal data types:")
display(df_media_raw.dtypes.to_frame("dtype"))

print("\nDataset preview:")
display(df_media_raw.head())

Original dataset shape: (257500, 4)

Original data types:


,dtype
id_media,int64
media_url,str
truth_id,int64
media_id,int64



Dataset preview:


,id_media,media_url,truth_id,media_id
0,2,https://static-assets-1.truthsocial.com/tmtg:p...,37,2
1,2,https://static-assets-1.truthsocial.com/tmtg:p...,384189,2
2,2,https://static-assets-1.truthsocial.com/tmtg:p...,508873,2
3,5,https://static-assets-1.truthsocial.com/tmtg:p...,40,5
4,5,https://static-assets-1.truthsocial.com/tmtg:p...,386893,5


In [2]:
# Count exact duplicate records
exact_duplicates = df_media_clean.duplicated().sum()

# Count duplicated media-post relationships
duplicate_pairs = df_media_clean.duplicated(
    subset=["id_media", "truth_id"]
).sum()

print("DUPLICATE VALIDATION")
print("-" * 40)

print("Exact duplicate records:", exact_duplicates)
print("Duplicate media-post pairs:", duplicate_pairs)

# Check for additional pair duplicates
additional_pair_duplicates = (
    duplicate_pairs - exact_duplicates
)

print(
    "Additional pair duplicates:",
    additional_pair_duplicates
)

# Remove exact duplicates
df_media_clean = (
    df_media_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

# Validate duplicate removal
remaining_duplicates = df_media_clean.duplicated().sum()

remaining_duplicate_pairs = df_media_clean.duplicated(
    subset=["id_media", "truth_id"]
).sum()

print("\nOriginal records:", len(df_media_raw))
print("Duplicates removed:", exact_duplicates)
print("Cleaned records:", len(df_media_clean))

print("Remaining exact duplicates:", remaining_duplicates)
print(
    "Remaining duplicate media-post pairs:",
    remaining_duplicate_pairs
)

# Structural validation
assert remaining_duplicates == 0

assert len(df_media_clean) == (
    len(df_media_raw) - exact_duplicates
)

DUPLICATE VALIDATION
----------------------------------------
Exact duplicate records: 4
Duplicate media-post pairs: 4
Additional pair duplicates: 0

Original records: 257500
Duplicates removed: 4
Cleaned records: 257496
Remaining exact duplicates: 0
Remaining duplicate media-post pairs: 0


In [3]:
# Define identifier columns
id_columns = [
    "id_media",
    "media_id",
    "truth_id"
]

# Convert identifiers to nullable integers
for col in id_columns:
    df_media_clean[col] = pd.to_numeric(
        df_media_clean[col],
        errors="coerce"
    ).astype("Int64")

# Check missing identifiers
missing_ids = df_media_clean[id_columns].isna().sum()

# Check non-positive identifiers
non_positive_ids = (
    df_media_clean[id_columns]
    .le(0)
    .sum()
)

# Verify consistency between media identifiers
id_mismatches = (
    df_media_clean["id_media"]
    != df_media_clean["media_id"]
).fillna(False).sum()

# Check duplicate media-post relationships
duplicate_pairs = df_media_clean.duplicated(
    subset=["id_media", "truth_id"]
).sum()

# Build validation report
identifier_validation = pd.DataFrame({
    "Check": [
        "Missing identifiers",
        "Non-positive identifiers",
        "Media ID mismatches",
        "Duplicate media-post relationships"
    ],
    "Count": [
        missing_ids.sum(),
        non_positive_ids.sum(),
        id_mismatches,
        duplicate_pairs
    ]
})

print("IDENTIFIER VALIDATION")
print("-" * 45)

display(identifier_validation)

print("\nData types after conversion:")
display(df_media_clean.dtypes.to_frame("dtype"))

print("\nDataset shape:")
print(df_media_clean.shape)

IDENTIFIER VALIDATION
---------------------------------------------


,Check,Count
0,Missing identifiers,0
1,Non-positive identifiers,0
2,Media ID mismatches,0
3,Duplicate media-post relationships,0



Data types after conversion:


,dtype
id_media,Int64
media_url,str
truth_id,Int64
media_id,Int64



Dataset shape:
(257496, 4)


In [4]:
from urllib.parse import urlsplit
import pandas as pd

# Define URL validation function
def is_valid_media_url(value):

    if pd.isna(value):
        return False

    if not isinstance(value, str):
        return False

    value = value.strip()

    if not value:
        return False

    try:
        parsed = urlsplit(value)

        return (
            parsed.scheme.lower() in ("http", "https")
            and bool(parsed.hostname)
            and not any(
                char.isspace()
                for char in parsed.hostname
            )
        )

    except ValueError:
        return False


# Validate media URLs
url_validity = df_media_clean["media_url"].apply(
    is_valid_media_url
)

# Check missing and empty URLs
missing_urls = df_media_clean["media_url"].isna().sum()

empty_urls = (
    df_media_clean["media_url"]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
    .sum()
)

# Count invalid URL structures
invalid_urls = (~url_validity).sum()

# Build validation summary
url_validation = pd.DataFrame({
    "Check": [
        "Missing media URLs",
        "Empty media URLs",
        "Invalid media URL structures",
        "Valid media URLs"
    ],
    "Count": [
        missing_urls,
        empty_urls,
        invalid_urls,
        url_validity.sum()
    ]
})

print("MEDIA URL VALIDATION")
print("-" * 45)

display(url_validation)

print("\nExamples of invalid media URLs:")

display(
    df_media_clean.loc[
        ~url_validity,
        ["id_media", "media_url", "truth_id"]
    ].head(20)
)

MEDIA URL VALIDATION
---------------------------------------------


,Check,Count
0,Missing media URLs,0
1,Empty media URLs,0
2,Invalid media URL structures,0
3,Valid media URLs,257496



Examples of invalid media URLs:


,id_media,media_url,truth_id


In [5]:
# Count distinct URLs associated with each media ID
urls_per_media_id = (
    df_media_clean
    .groupby("id_media")["media_url"]
    .nunique()
)

# Count distinct media IDs associated with each URL
ids_per_media_url = (
    df_media_clean
    .groupby("media_url")["id_media"]
    .nunique()
)

# Identify inconsistent mappings
inconsistent_media_ids = (
    urls_per_media_id > 1
).sum()

inconsistent_media_urls = (
    ids_per_media_url > 1
).sum()

# Check consistency between redundant identifiers
id_mismatches = (
    df_media_clean["id_media"]
    != df_media_clean["media_id"]
).fillna(False).sum()

# Build validation summary
mapping_validation = pd.DataFrame({
    "Check": [
        "Media IDs associated with multiple URLs",
        "Media URLs associated with multiple IDs",
        "Media identifier mismatches"
    ],
    "Count": [
        inconsistent_media_ids,
        inconsistent_media_urls,
        id_mismatches
    ]
})

print("MEDIA MAPPING CONSISTENCY")
print("-" * 45)

display(mapping_validation)

print("\nInconsistent media IDs:")
display(
    urls_per_media_id[
        urls_per_media_id > 1
    ].head(10)
)

print("\nInconsistent media URLs:")
display(
    ids_per_media_url[
        ids_per_media_url > 1
    ].head(10)
)

MEDIA MAPPING CONSISTENCY
---------------------------------------------


,Check,Count
0,Media IDs associated with multiple URLs,0
1,Media URLs associated with multiple IDs,0
2,Media identifier mismatches,0



Inconsistent media IDs:


Series([], Name: media_url, dtype: int64)


Inconsistent media URLs:


Series([], Name: id_media, dtype: int64)

In [6]:
# Define path to cleaned Truths dataset
TRUTHS_CLEAN_PATH = DATA_DIR / "truths_clean.csv"

if not TRUTHS_CLEAN_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {TRUTHS_CLEAN_PATH}"
    )

# Load post identifiers only
df_truths_ids = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)

# Extract unique post identifiers
valid_truth_ids = pd.Index(
    df_truths_ids["id"].dropna().unique()
)

# Check referential integrity
truth_exists = df_media_clean["truth_id"].isin(
    valid_truth_ids
)

# Count matched and unmatched relationships
matched_relationships = truth_exists.sum()
unmatched_relationships = (~truth_exists).sum()

# Count distinct unmatched post IDs
unmatched_post_ids = (
    df_media_clean.loc[
        ~truth_exists, "truth_id"
    ].nunique()
)

# Build summary
referential_summary = pd.DataFrame({
    "Check": [
        "Total media-post relationships",
        "Matched relationships",
        "Unmatched relationships",
        "Distinct unmatched post IDs"
    ],
    "Count": [
        len(df_media_clean),
        matched_relationships,
        unmatched_relationships,
        unmatched_post_ids
    ]
})

print("REFERENTIAL INTEGRITY — MAIN TRUTHS")
print("-" * 45)

display(referential_summary)

print(
    "\nUnmatched percentage:",
    round(
        unmatched_relationships
        / len(df_media_clean) * 100,
        2
    ),
    "%"
)

print("\nExamples of unmatched relationships:")

display(
    df_media_clean.loc[
        ~truth_exists,
        ["id_media", "media_url", "truth_id"]
    ].head(10)
)

REFERENTIAL INTEGRITY — MAIN TRUTHS
---------------------------------------------


,Check,Count
0,Total media-post relationships,257496
1,Matched relationships,155188
2,Unmatched relationships,102308
3,Distinct unmatched post IDs,98641



Unmatched percentage: 39.73 %

Examples of unmatched relationships:


,id_media,media_url,truth_id
455,67,https://static-assets-1.truthsocial.com/tmtg:p...,112
456,67,https://static-assets-1.truthsocial.com/tmtg:p...,114431
457,67,https://static-assets-1.truthsocial.com/tmtg:p...,120836
458,67,https://static-assets-1.truthsocial.com/tmtg:p...,158929
459,67,https://static-assets-1.truthsocial.com/tmtg:p...,162173
460,67,https://static-assets-1.truthsocial.com/tmtg:p...,152209
461,67,https://static-assets-1.truthsocial.com/tmtg:p...,173087
462,67,https://static-assets-1.truthsocial.com/tmtg:p...,176756
463,67,https://static-assets-1.truthsocial.com/tmtg:p...,196558
464,67,https://static-assets-1.truthsocial.com/tmtg:p...,203483


In [10]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data"
)

TRUTHS_CLEAN_PATH = (
    BASE_PATH / "Project_Data" / "truths_clean.csv"
)

ADDITIONAL_PATH = (
    BASE_PATH / "Additional_Data" / "additional_truths.arff"
)

# Verify input files
for path in [TRUTHS_CLEAN_PATH, ADDITIONAL_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

# Load the cleaned main corpus
# Preserve external IDs as strings to avoid precision loss
df_truths = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    dtype={
        "external_id": "string",
        "url": "string"
    },
    low_memory=False
)

print("MAIN CORPUS")
print("-" * 40)
print("Shape:", df_truths.shape)

print("\nADDITIONAL ARFF")
print("-" * 40)
print("File:", ADDITIONAL_PATH)
print("Size (MB):", round(
    ADDITIONAL_PATH.stat().st_size / 1024**2, 2
))

print("\nMAIN CORPUS COLUMNS")
print(df_truths.columns.tolist())

print("\nMain corpus successfully loaded.")


MAIN CORPUS
----------------------------------------
Shape: (1038821, 13)

ADDITIONAL ARFF
----------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Additional_Data\additional_truths.arff
Size (MB): 13.13

MAIN CORPUS COLUMNS
['id', 'timestamp', 'time_scraped', 'is_retruth', 'is_reply', 'author', 'like_count', 'retruth_count', 'reply_count', 'text', 'external_id', 'url', 'truth_retruthed']

Main corpus successfully loaded.


In [11]:

# Load additional_truths.arff using the validated parsing strategy

additional_columns = [
    "id",
    "timestamp",
    "text",
    "url",
    "reply_count",
    "retruth_count",
    "like_count",
    "external_id",
    "author",
    "time_scraped"
]

records = []
data_started = False

with open(ADDITIONAL_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if line.lower() == "@data":
            data_started = True
            continue

        if not data_started or not line or line.startswith("%"):
            continue

        # First two attributes from the left
        left = line.split(",", 2)

        # Remaining seven attributes from the right
        right = left[2].rsplit(",", 7)

        row = [left[0], left[1]] + right

        if len(row) != len(additional_columns):
            raise ValueError(
                f"Invalid ARFF record with {len(row)} fields"
            )

        records.append(row)

df_additional = pd.DataFrame(
    records,
    columns=additional_columns
)

# Remove enclosing quotes without changing internal text
def remove_enclosing_quotes(value):
    value = value.strip()

    if len(value) >= 2 and value[0] == value[-1]:
        if value[0] in ("'", '"'):
            return value[1:-1]

    return value

for col in df_additional.columns:
    df_additional[col] = (
        df_additional[col].map(remove_enclosing_quotes)
    )

# Normalize identifiers without converting large IDs to floats
for col in ["id", "external_id"]:
    df_additional[col] = (
        df_additional[col].astype("string").str.strip()
    )

print("ADDITIONAL TRUTHS LOADING")
print("-" * 45)

print("Shape:", df_additional.shape)

print("\nIDENTIFIER VALIDATION")

id_match_mask = (
    df_additional["id"]
    .eq(df_additional["external_id"])
    .fillna(False)
)

print(
    "ID / external_id matches:",
    f"{id_match_mask.sum():,}"
)

print(
    "ID / external_id mismatches:",
    f"{(~id_match_mask).sum():,}"
)

print(
    "Duplicated external IDs:",
    f"{df_additional['external_id'].duplicated().sum():,}"
)

print("\nSAMPLE RECORDS")
display(df_additional.head())


ADDITIONAL TRUTHS LOADING
---------------------------------------------
Shape: (35222, 10)

IDENTIFIER VALIDATION
ID / external_id matches: 35,222
ID / external_id mismatches: 0
Duplicated external IDs: 0

SAMPLE RECORDS


,id,timestamp,text,url,reply_count,retruth_count,like_count,external_id,author,time_scraped
0,117323511508375196,2026-09-24 01:51:54,,https://truthsocial.com/@realDonaldTrump/11732...,792.0,1849.0,7259.0,117323511508375196,27.0,2026-09-20 14:00:00.000000
1,117323155721195611,2026-09-24 00:21:25,"The story that the White House, or me, want to...",https://truthsocial.com/@realDonaldTrump/11732...,1329.0,3534.0,12560.0,117323155721195611,27.0,2026-09-20 14:00:00.000000
2,117323148147220269,2026-09-24 00:19:29,BIG DAY FOR BOEING AND AMERICAN MANUFACTURING!...,https://truthsocial.com/@realDonaldTrump/11732...,510.0,2714.0,10677.0,117323148147220269,27.0,2026-09-20 14:00:00.000000
3,117322092055919832,2026-09-23 19:50:55,,https://truthsocial.com/@realDonaldTrump/11732...,1344.0,2740.0,11287.0,117322092055919832,27.0,2026-09-20 14:00:00.000000
4,117322065119445871,2026-09-23 19:44:04,"The Great Gary Player, Bryson DeChambeau, Mika...",https://truthsocial.com/@realDonaldTrump/11732...,505.0,2156.0,10284.0,117322065119445871,27.0,2026-09-20 14:00:00.000000


In [12]:
# Verify that the Additional Truths dataset is loaded
if "df_additional" not in globals():
    raise NameError(
        "df_additional is not loaded. "
        "Run the validated ARFF loading cell first."
    )

# Prepare identifier columns
df_additional_ref = df_additional[
    ["id", "external_id", "url"]
].copy()

for col in ["id", "external_id"]:
    df_additional_ref[col] = pd.to_numeric(
        df_additional_ref[col],
        errors="coerce"
    ).astype("Int64")

# Load main Truths identifiers
df_truths_keys = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id", "external_id", "url"],
    dtype={
        "id": "Int64",
        "external_id": "string",
        "url": "string"
    },
    low_memory=False
)

df_truths_keys["external_id"] = pd.to_numeric(
    df_truths_keys["external_id"],
    errors="coerce"
).astype("Int64")

# Identify missing internal post IDs
missing_post_ids = pd.Index(
    df_media_clean.loc[
        ~truth_exists, "truth_id"
    ].dropna().unique()
)

# Compare with Additional Truths identifiers
additional_internal_ids = pd.Index(
    df_additional_ref["id"].dropna().unique()
)

additional_external_ids = pd.Index(
    df_additional_ref["external_id"].dropna().unique()
)

# Direct ID overlap
matches_with_id = missing_post_ids.intersection(
    additional_internal_ids
)

matches_with_external_id = missing_post_ids.intersection(
    additional_external_ids
)

# External ID overlap with main Truths
main_external_ids = pd.Index(
    df_truths_keys["external_id"].dropna().unique()
)

shared_external_ids = main_external_ids.intersection(
    additional_external_ids
)

# Build comparison report
identifier_comparison = pd.DataFrame({
    "Check": [
        "Distinct unmatched media post IDs",
        "Matches with Additional Truths id",
        "Matches with Additional Truths external_id",
        "Shared external IDs between Truths datasets"
    ],
    "Count": [
        len(missing_post_ids),
        len(matches_with_id),
        len(matches_with_external_id),
        len(shared_external_ids)
    ]
})

print("ADDITIONAL TRUTHS IDENTIFIER COMPARISON")
print("-" * 50)

display(identifier_comparison)

print("\nIDENTIFIER RANGES")
print("-" * 50)

ranges = pd.DataFrame({
    "Identifier": [
        "media.truth_id",
        "truths.id",
        "truths.external_id",
        "additional.id",
        "additional.external_id"
    ],
    "Minimum": [
        df_media_clean["truth_id"].min(),
        df_truths_keys["id"].min(),
        df_truths_keys["external_id"].min(),
        df_additional_ref["id"].min(),
        df_additional_ref["external_id"].min()
    ],
    "Maximum": [
        df_media_clean["truth_id"].max(),
        df_truths_keys["id"].max(),
        df_truths_keys["external_id"].max(),
        df_additional_ref["id"].max(),
        df_additional_ref["external_id"].max()
    ]
})

display(ranges)

print("\nExamples of shared external IDs:")
print(shared_external_ids[:10].tolist())

ADDITIONAL TRUTHS IDENTIFIER COMPARISON
--------------------------------------------------


,Check,Count
0,Distinct unmatched media post IDs,98641
1,Matches with Additional Truths id,0
2,Matches with Additional Truths external_id,0
3,Shared external IDs between Truths datasets,32



IDENTIFIER RANGES
--------------------------------------------------


,Identifier,Minimum,Maximum
0,media.truth_id,37,782306
1,truths.id,18,2383524
2,truths.external_id,107760498125955210,113392815931039883
3,additional.id,108211822140637685,117323511508375196
4,additional.external_id,108211822140637685,117323511508375196



Examples of shared external IDs:
[109110471933646660, 109110378607366886, 109105126982139161, 109105107965281984, 109102258492682585, 109111035929718455, 109096508463033991, 109093444861506227, 109102004910960727, 109082720069591446]


In [13]:
# Final Data Quality Assessment

id_columns = ["id_media", "media_id", "truth_id"]

missing_values = df_media_clean.isna().sum().sum()

invalid_ids = (
    df_media_clean[id_columns]
    .le(0)
    .sum()
    .sum()
)

duplicate_records = df_media_clean.duplicated().sum()

duplicate_pairs = df_media_clean.duplicated(
    subset=["id_media", "truth_id"]
).sum()

id_mismatches = (
    df_media_clean["id_media"]
    != df_media_clean["media_id"]
).fillna(False).sum()

invalid_urls = (
    ~df_media_clean["media_url"].apply(
        is_valid_media_url
    )
).sum()

inconsistent_media_ids = (
    df_media_clean.groupby("id_media")["media_url"]
    .nunique()
    .gt(1)
    .sum()
)

inconsistent_media_urls = (
    df_media_clean.groupby("media_url")["id_media"]
    .nunique()
    .gt(1)
    .sum()
)

unmatched_references = (
    ~df_media_clean["truth_id"].isin(valid_truth_ids)
).sum()

# Build quality report
quality_report = pd.DataFrame({
    "Check": [
        "Missing values",
        "Invalid identifiers",
        "Duplicate records",
        "Duplicate media-post relationships",
        "Media identifier mismatches",
        "Invalid media URLs",
        "Media IDs associated with multiple URLs",
        "Media URLs associated with multiple IDs",
        "Unmatched post references"
    ],
    "Count": [
        missing_values,
        invalid_ids,
        duplicate_records,
        duplicate_pairs,
        id_mismatches,
        invalid_urls,
        inconsistent_media_ids,
        inconsistent_media_urls,
        unmatched_references
    ]
})

print("FINAL DATA QUALITY ASSESSMENT")
print("-" * 45)

display(quality_report)

print("\nDATASET COMPARISON")
print("-" * 45)

print("Original records:", len(df_media_raw))
print("Duplicates removed:", exact_duplicates)
print("Final records:", len(df_media_clean))

# Validate cleaning results
assert missing_values == 0
assert invalid_ids == 0
assert duplicate_records == 0
assert duplicate_pairs == 0
assert id_mismatches == 0
assert invalid_urls == 0
assert inconsistent_media_ids == 0
assert inconsistent_media_urls == 0

assert len(df_media_clean) == (
    len(df_media_raw) - exact_duplicates
)

print("\nAll internal data quality checks passed.")

FINAL DATA QUALITY ASSESSMENT
---------------------------------------------


,Check,Count
0,Missing values,0
1,Invalid identifiers,0
2,Duplicate records,0
3,Duplicate media-post relationships,0
4,Media identifier mismatches,0
5,Invalid media URLs,0
6,Media IDs associated with multiple URLs,0
7,Media URLs associated with multiple IDs,0
8,Unmatched post references,102308



DATASET COMPARISON
---------------------------------------------
Original records: 257500
Duplicates removed: 4
Final records: 257496

All internal data quality checks passed.


In [14]:
# Preserve original schema
df_media_final = df_media_clean[
    ["id_media", "media_url", "truth_id", "media_id"]
].copy()

# Define export path
MEDIA_CLEAN_PATH = DATA_DIR / "media_clean.csv"

# Export cleaned dataset
df_media_final.to_csv(
    MEDIA_CLEAN_PATH,
    index=False
)

# Final validation
assert df_media_final.shape == (257496, 4)

print("EXPORT COMPLETED")
print("-" * 40)

print("File:", MEDIA_CLEAN_PATH)

print("\nFinal dataset shape:")
print(df_media_final.shape)

print("\nFinal dataset preview:")
display(df_media_final.head())

EXPORT COMPLETED
----------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\media_clean.csv

Final dataset shape:
(257496, 4)

Final dataset preview:


,id_media,media_url,truth_id,media_id
0,2,https://static-assets-1.truthsocial.com/tmtg:p...,37,2
1,2,https://static-assets-1.truthsocial.com/tmtg:p...,384189,2
2,2,https://static-assets-1.truthsocial.com/tmtg:p...,508873,2
3,5,https://static-assets-1.truthsocial.com/tmtg:p...,40,5
4,5,https://static-assets-1.truthsocial.com/tmtg:p...,386893,5
